# Multiple State Schemas

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn how to use `InputState`, `OutputState`, and `OverallState` to control what callers see, and how to use private internal channels for intermediate data.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define Separate Schemas

Create `InputState`, `OutputState`, and `OverallState` as separate TypedDicts.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str

class OverallState(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]
    research_notes: str

print("InputState fields:", list(InputState.__annotations__))
print("OutputState fields:", list(OutputState.__annotations__))
print("OverallState fields:", list(OverallState.__annotations__))

## 4. Build the Graph with Schema Parameters

Pass `input=InputState` and `output=OutputState` to `StateGraph`.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def research(state: OverallState) -> dict:
    response = llm.invoke([
        ("system", "Research the question and provide detailed notes."),
        ("human", state["question"]),
    ])
    return {
        "messages": [response],
        "research_notes": response.content,
    }

def synthesize(state: OverallState) -> dict:
    response = llm.invoke([
        ("system", "Synthesize a concise answer from the research notes."),
        ("human", state["research_notes"]),
    ])
    return {"answer": response.content}

graph = StateGraph(
    OverallState,
    input=InputState,
    output=OutputState,
)
graph.add_node("research", research)
graph.add_node("synthesize", synthesize)
graph.add_edge(START, "research")
graph.add_edge("research", "synthesize")
graph.add_edge("synthesize", END)

app = graph.compile()

## 5. Test — Only OutputState Fields Returned

The caller sends `InputState` fields and receives only `OutputState` fields.

In [ ]:
result = app.invoke({"question": "What are reducers in LangGraph?"})
print(f"Result keys: {list(result.keys())}")
print(f"\nAnswer:\n{result['answer']}")

In [ ]:
print("'messages' in result:", "messages" in result)
print("'research_notes' in result:", "research_notes" in result)
print("'question' in result:", "question" in result)

## 6. Comparison: Single Schema vs Multiple Schemas

Without separate schemas, all internal fields leak to the caller.

In [ ]:
graph_single = StateGraph(OverallState)
graph_single.add_node("research", research)
graph_single.add_node("synthesize", synthesize)
graph_single.add_edge(START, "research")
graph_single.add_edge("research", "synthesize")
graph_single.add_edge("synthesize", END)

app_single = graph_single.compile()

result_single = app_single.invoke({"question": "What are reducers in LangGraph?"})
print(f"Single schema result keys: {list(result_single.keys())}")

## What You Learned

1. **`InputState`** defines what the caller provides; **`OutputState`** defines what they receive
2. **`OverallState`** is the internal superset used by all nodes
3. Use `StateGraph(OverallState, input=InputState, output=OutputState)` to wire them up
4. Fields only in `OverallState` are **private internal channels** invisible to the caller
5. Multiple schemas keep your graph's public API clean and predictable